In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

TEST_HAND_FILE = os.getenv('TEST_HAND_FILE')

In [ ]:
with open(TEST_HAND_FILE, 'r') as f:
    hand_data = f.readlines()

single_hand = hand_data[:20]

single_hand_dict = {} #complete data for a sinle hand, with all the info we want to extract
hand_dict = {} #id, stakes, board cards
player_dict = {} #one row per player: seat, position, starting stack, amount won/lost
actions_dict = {} #one row per action: street, player, action type, amount

for line in single_hand:
    if '=' in line:
        single_hand_dict[line.split('=')[0].strip()] = line.split('=')[1].strip()

# Hand Dict
hand_dict['id'] = single_hand_dict['hand']
hand_dict['stakes'] = single_hand_dict['blinds_or_straddles']
#hand_dict['board'] = single_hand_dict['board']

# Player Dict
player_ids = single_hand_dict['players'].split(',')
player_seats = single_hand_dict['seats'].split(',')
player_starting_stacks = single_hand_dict['starting_stacks'].split(',')
player_winnings = single_hand_dict['winnings'].split(',')
player_actions = single_hand_dict['actions'].split(',')

for i in range(len(player_ids)):
    player_dict[player_ids[i]] = {
        'seat': player_seats[i],
        'starting_stack': player_starting_stacks[i],
        'winnings': player_winnings[i]
    }

def translate_card_names(card_str):
    """ Translates the shorthand from PHH into english card names. 
    expects 2 character card strings, e.g. 'Ah' for Ace of Hearts
    """
    rank_translation = {
        '2': 'Two',
        '3': 'Three',
        '4': 'Four',
        '5': 'Five',
        '6': 'Six',
        '7': 'Seven',
        '8': 'Eight',
        '9': 'Nine',
        'T': 'Ten',
        'J': 'Jack',
        'Q': 'Queen',
        'K': 'King',
        'A': 'Ace'
    }
    suit_translation = {
        'h': 'Hearts',
        'd': 'Diamonds',
        'c': 'Clubs',
        's': 'Spades'
    }
    rank = card_str[0]
    suit = card_str[1]
    return f"{rank_translation[rank]} of {suit_translation[suit]}"

action_id = 1

for action in player_actions:
    action = action.replace('[', '').replace(']', '') #Remove brackets from actions
    action = action.replace("'", "") #Remove spaces from actions
    action = action.strip() #Remove leading/trailing whitespace
    action_parts = action.split(' ')
    action_description = ''
    
    if action_parts[0] == 'd': #Dealer action:
        role = 'Dealer'
        if action_parts[1] == 'dh': #Dealing hole cards
            action = 'Deal Hole Cards'
            action_description = f'{role} deals hole cards to player {action_parts[2]}'
        elif action_parts[1] == 'db': #Dealing board cards
            action = 'Deal Board Cards'
            cards = action_parts[2]
            if len(cards) % 2 != 0:
                raise ValueError(f"Invalid card string: {cards}")
            else:
                card_list = [translate_card_names(cards[i:i+2]) for i in range(0, len(cards), 2)] #Check output of this to make sure it works as expected
            action_description = f'{role} deals board cards'
        else:
            action = 'Unknown Dealer Action'

    if action_parts[0].startswith('p'): #Player action:
        role = 'Player'
        player_id = action_parts[0][-1:] #Remove the 'p' from the player id
        if action_parts[1] == 'cc': #Call
            action = 'Calls'
        elif action_parts[1] == 'cbr': #Raise
            action = f'Raises to {action_parts[2]}'
        elif action_parts[1] == 'f': #Fold
            action = 'Folds'
        else:
            action = 'Unknown Player Action'
        action_description = f'{role} {player_id} {action}'
    
    actions_dict[action_id] = action_description
    action_id += 1

print(hand_dict)
print(player_dict)
print(actions_dict)